In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

isCuda = try
    success(`nvidia-smi`)
catch
    false
end
if isCuda
    println("CUDA is available, using GPU acceleration.")
    using CUDA
end

using bslLD
bslLD.greet()

if isCuda
    println("Setting backend to CUDA.")
    bslLD.use_cuda!()
else
    println("CUDA not available, using CPU.")
end


In [ ]:
using CairoMakie

# 1D Vacuum Maxwell Example


In [ ]:
function run_simulation!(ax::Axis, c::Real)
    nx = 512
    dt = 0.01
    nt = 500
    Lx = 12.0
    grid = bslLD.Grid([0.0], [Lx], [nx], 1)
    simTime = bslLD.SimulationTime(dt, nt*dt)

    params = bslLD.VacuumMaxwellParams(c=c, ϵ0=2.0, μ0=1.0)  # fix: use c argument

    Ey0 = rand(Float64, nx)
    E = bslLD.VectorField([zeros(nx), Ey0, zeros(nx)])
    B = bslLD.VectorField([zeros(nx), zeros(nx), zeros(nx)])

    E_record = zeros(nx, nt+1)
    while bslLD.continue_advection(simTime, true)
        bslLD.step_maxwell_cn!(E, B, grid; dt=dt, params=params)
        E_record[:, simTime.step+1] = E[2].data
        bslLD.advance!(simTime)
    end

    fft_result = bslLD.fft(E_record)

    nk  = nx ÷ 2          # number of wavenumber bins (rows)
    nω  = nt ÷ 2          # number of frequency bins (cols)

    dk  = 2π / Lx
    dω  = 2π / (nt * dt)
    k_axis = (0:nk-1) .* dk
    ω_axis = (0:nω-1) .* dω

    heatmap!(ax, k_axis, ω_axis,              # fix: plot onto provided Axis
             log.(abs.(fft_result[1:nk, 1:nω]) .+ 1e-10);  # +eps avoids log(0)
             colormap=:viridis)
end


In [ ]:

c_values = [1.0, 2.0, 4.0]
fig = Figure(resolution=(1200, 400))
for (i, c) in enumerate(c_values)
    ax = Axis(fig[1,i],
              title  = "c = $c",
              xlabel = "wavenumber k",
              ylabel = "frequency ω")
    run_simulation!(ax, c)
end
fig

# 2D Vacuum Maxwell Example

This notebook sets up a 2D periodic grid with 3-component electromagnetic fields, initializes a localized TM-like pulse, advances it with the vacuum Crank-Nicolson Maxwell solver, plots diagnostics, and records a GIF of the outward travelling wave.

In [ ]:
nx = 256
ny = 512
dt = 0.04
nt = 300
Lx = 12.0
Ly = 24.0

grid = bslLD.Grid([0.0, 0.0], [Lx, Ly], [nx, ny], 2)
simTime = bslLD.SimulationTime(dt, nt*dt)
x = collect(grid.xaxes[1])
y = collect(grid.xaxes[2])
times = collect(0.0:dt:nt * dt)
params = bslLD.VacuumMaxwellParams(c=1.0, ϵ0=1.0, μ0=1.0)

xc = Lx / 2
yc = Ly / 4
σ = 0.7
ϵr = 1e-8
amplitude = 1.0

In [ ]:
X = reshape(x, :, 1) .* ones(1, ny)
Y = ones(nx, 1) .* reshape(y, 1, :)
dx = X .- xc
dy = Y .- yc
r = sqrt.(dx .^ 2 .+ dy .^ 2 .+ ϵr)

Ez0 = amplitude .* exp.(-(r .^ 2) ./ (2σ^2))
envelope = Ez0 ./ params.c

# Tangential in-plane magnetic field for an outward-travelling TM-like pulse.
Bx0 = .-(dy ./ r) .* envelope
By0 =  (dx ./ r) .* envelope

E = bslLD.VectorField([
    zeros(nx, ny),
    zeros(nx, ny),
    Ez0,
])

B = bslLD.VectorField([
    Bx0,
    By0,
    zeros(nx, ny),
])

In [ ]:
X = reshape(x, :, 1) .* ones(1, ny)
Y = ones(nx, 1) .* reshape(y, 1, :)

# Center of the wave packet
xc = Lx / 2
yc = Ly / 4

dy_env = Y .- yc  # signed distance along propagation direction
dx_env = X .- xc  # signed distance transverse to propagation

# Gaussian envelope centered at yc
Ez0 = amplitude .* exp.(-(dx_env .^ 2/10 .+ dy_env .^ 2) ./ (2σ^2)) .* sin.(10 * (dy_env .- 0.5) ./ 1.0)

# Plane wave travelling in +y: k̂ = ŷ
# For a TM wave with E = Ez ẑ and k̂ = ŷ:
#   B = (k̂ × Ê) / c = (ŷ × ẑ) / c * Ez = x̂ / c * Ez
# So Bx = Ez / c, By = 0
Bx0 =  Ez0 ./ params.c
By0 =  zeros(nx, ny)

E = bslLD.VectorField([
    zeros(nx, ny),
    zeros(nx, ny),
    Ez0,
])

B = bslLD.VectorField([
    Bx0,
    By0,
    zeros(nx, ny),
])

In [ ]:
heatmap(x, y, Ez0'; colormap=:viridis, colorrange=(-amplitude, amplitude))

In [ ]:
# --- Double slit wall ---
# --- Double slit wall ---
wall_y = Ly / 2
wall_j = argmin(abs.(y .- wall_y))
wall_thickness = 3

# Slit parameters (in units of your domain)
slit_width    = 0.8         # ~1.3λ → good for diffraction
slit_half_sep = 2.0         # ~3.2λ → good for interference
slit1_centre  = Lx / 2 - slit_half_sep
slit2_centre  = Lx / 2 + slit_half_sep

slit_mask = ones(Float64, nx, ny)      # 1 = free, 0 = conductor

for dj in 0:wall_thickness-1
    jj = wall_j + dj
    for i in 1:nx
        xi = x[i]
        in_slit1 = abs(xi - slit1_centre) <= slit_width
        in_slit2 = abs(xi - slit2_centre) <= slit_width
        if !(in_slit1 || in_slit2)
            slit_mask[i, jj] = 0.0
        end
    end
end

# --- Absorbing boundary (cosine-squared ramp) ---
pml_depth = 20                         # cells from each edge
pml_mask  = ones(Float64, nx, ny)

for j in 1:ny, i in 1:nx
    di = min(i - 1, nx - i)           # distance from x-edge
    dj = min(j - 1, ny - j)           # distance from y-edge
    d  = min(di, dj, pml_depth)       # effective depth
    if d < pml_depth
        α = cos(π/2 * d / pml_depth)  # 1 at interior, 0 at boundary
        pml_mask[i, j] = α^4          # raise power for stronger damping
    end
end

In [ ]:
function apply_masks!(E, B, slit_mask, pml_mask)
    combined = slit_mask .* pml_mask
    E[3].data .*= combined        # Ez
    B[1].data .*= combined        # Bx
    B[2].data .*= combined        # By
end

In [ ]:
Ez_history = zeros(length(times), nx, ny)
Bx_history = zeros(length(times), nx, ny)
By_history = zeros(length(times), nx, ny)
energy = zeros(length(times))
divE_max = zeros(length(times))
divB_max = zeros(length(times))

function record_state!(slot, E, B, grid, params, Ez_history, Bx_history, By_history, energy, divE_max, divB_max)
    Ez_history[slot, :, :] .= E[3].data
    Bx_history[slot, :, :] .= B[1].data
    By_history[slot, :, :] .= B[2].data
    energy[slot] = bslLD.electromagnetic_energy(E, B; params=params)
    divE, divB = bslLD.maxwell_constraints(E, B, grid)
    divE_max[slot] = maximum(abs.(divE.data))
    divB_max[slot] = maximum(abs.(divB.data))
    return nothing
end

record_state!(1, E, B, grid, params, Ez_history, Bx_history, By_history, energy, divE_max, divB_max)

for n in 1:nt
    bslLD.step_maxwell_cn!(E, B, grid; dt=dt, params=params)
    apply_masks!(E, B, slit_mask, pml_mask)                  # ← add this
    record_state!(n + 1, E, B, grid, params, Ez_history, Bx_history, By_history, energy, divE_max, divB_max)
end

energy_rel = energy ./ energy[1]



println("Initial energy       = ", energy[1])
println("Final energy         = ", energy[end])
println("Relative energy drift = ", abs(energy[end] - energy[1]) / energy[1])
println("Max |div E| over run  = ", maximum(divE_max))
println("Max |div B| over run  = ", maximum(divB_max))
println("GIF written to        = ", gif_path)

In [ ]:
sample_ids = round.(Int, range(1, length(times), length=4))

fig = Figure(size=(1200, 900))

ax1 = Axis(fig[1, 1], title="E_z at t = $(round(times[sample_ids[1]], digits=2))", xlabel="x", ylabel="y")
hm1 = heatmap!(ax1, x, y, Ez_history[sample_ids[1], :, :]; colormap=:balance)
Colorbar(fig[1, 2], hm1, label="E_z")

ax2 = Axis(fig[1, 3], title="E_z at t = $(round(times[sample_ids[2]], digits=2))", xlabel="x", ylabel="y")
hm2 = heatmap!(ax2, x, y, Ez_history[sample_ids[2], :, :]; colormap=:balance)
Colorbar(fig[1, 4], hm2, label="E_z")

ax3 = Axis(fig[2, 1], title="E_z at t = $(round(times[sample_ids[3]], digits=2))", xlabel="x", ylabel="y")
hm3 = heatmap!(ax3, x, y, Ez_history[sample_ids[3], :, :]; colormap=:balance)
Colorbar(fig[2, 2], hm3, label="E_z")

ax4 = Axis(fig[2, 3], title="E_z at t = $(round(times[sample_ids[4]], digits=2))", xlabel="x", ylabel="y")
hm4 = heatmap!(ax4, x, y, Ez_history[sample_ids[4], :, :]; colormap=:balance)
Colorbar(fig[2, 4], hm4, label="E_z")

fig

In [ ]:
gif_path = joinpath(pwd(), "solve_maxwell_2d.gif")

fig_anim = Figure(size=(900, 900))
ax_anim = Axis(fig_anim[1, 1], 
    xlabel="x", ylabel="y",
    aspect = DataAspect()
)

frame = Observable(Ez_history[1, :, :])
time_obs = Observable(0.0)

# Initial colorrange from first frame
frame_lim = maximum(abs, Ez_history[1, :, :])
clim = Observable((-frame_lim, frame_lim))

hm_anim = heatmap!(ax_anim, x, y, frame; colormap=:balance, colorrange=clim)

xlims!(ax_anim, minimum(x), maximum(x))
ylims!(ax_anim, minimum(y), maximum(y))

record(fig_anim, gif_path, 1:length(times); framerate=20) do i
    ez_frame = Ez_history[i, :, :]
    lim = maximum(abs, ez_frame)
    lim = lim > 0 ? lim : 1.0   # guard against all-zero frames
    clim[] = (-lim, lim)
    frame[] = ez_frame
    time_obs[] = times[i]
end

display(MIME("text/html"), "<img src=\"$(gif_path)\">")